## Changelog
- parent: 20260507_220446_b5313d5e
- change: switch model to XGBRegressor with tree_method='hist' and
    enable_categorical=True. Same preprocessing as the LightGBM attempt
    (AmesNAImputer + size + temporal + CategoryCaster, no AmesEncoder, no
    nbhd_te). Grid focuses on XGBoost's distinctive levers: reg_lambda (L2
    on leaf weights, which neither sklearn GBR nor LightGBM expose as a
    first-class default), max_depth, min_child_weight.
- hypothesis: LightGBM regressed (CV 0.1209) vs sklearn GBM tuned (0.1175)
    because leaf-wise growth overfits 1455-row training. XGBoost defaults
    to level-wise (more conservative on small data) and adds explicit L2
    regularization. If the diagnosis is right, XGBoost should beat LightGBM
    here; whether it also beats the sklearn GBM tuned is open — that would
    require L2 to add real signal that subsample=0.8 alone didn't capture.

In [ ]:
import sys
import numpy as np
import pandas as pd

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

    if str(comp_dir) not in sys.path:
        sys.path.insert(0, str(comp_dir))

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")

In [ ]:
# see eda-TotalSF.ipynb — drop the mega-house outliers (TotalSF > 7000)
_outlier_mask = (
    train_data_raw["TotalBsmtSF"]
    + train_data_raw["1stFlrSF"]
    + train_data_raw["2ndFlrSF"]
) > 7000
train_data = train_data_raw.loc[~_outlier_mask].reset_index(drop=True)

# Drop Id (row identifier, no signal) and SalePrice (target). Everything else
# goes through preprocessing.
DROP = ["Id", "SalePrice"]
X      = train_data.drop(columns=DROP, errors="ignore").copy()
X_test = test_data_raw.drop(columns=DROP, errors="ignore").copy()
y      = np.log1p(train_data["SalePrice"])

# MSSubClass is a nominal int code — cast to string so the encoder treats it
# as a category rather than an ordered number.
X["MSSubClass"]      = X["MSSubClass"].astype(str)
X_test["MSSubClass"] = X_test["MSSubClass"].astype(str)

# Auto-detect num/cat from train; apply the same split to test.
NUMERIC     = X.select_dtypes(include="number").columns.tolist()
CATEGORICAL = X.select_dtypes(exclude="number").columns.tolist()
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical = {len(X.columns)} total")

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import GridSearchCV, KFold

from xgboost import XGBRegressor

from utils.ames_sklearn_pipeline import AmesNAImputer
from utils.ames_feature_engineering import add_size_features, add_temporal_features


class CategoryCaster(BaseEstimator, TransformerMixin):
    """Pin the categorical vocabulary at fit-time and replay it at transform.

    Without this, train and val DataFrames would each call .astype('category')
    independently and end up with different categorical codes for the same
    string value — XGBoost's native categorical path would then map test
    rows through the wrong codes.
    """

    def fit(self, X, y=None):
        self.vocab_ = {}
        cat_like = X.select_dtypes(include=["object", "string", "category"]).columns
        for c in cat_like:
            self.vocab_[c] = sorted(X[c].dropna().astype(str).unique())
        return self

    def transform(self, X):
        out = X.copy()
        for c, cats in self.vocab_.items():
            if c in out.columns:
                out[c] = pd.Categorical(out[c].astype(object), categories=cats)
        return out


pipe = Pipeline([
    ("na",       AmesNAImputer()),
    ("size",     FunctionTransformer(
                     add_size_features,
                     kw_args={"drop_originals": True},
                 )),
    ("fe",       FunctionTransformer(
                     add_temporal_features,
                     kw_args={"drop_originals": True},
                 )),
    ("cat_cast", CategoryCaster()),
    ("model",    XGBRegressor(
                     tree_method="hist",
                     enable_categorical=True,
                     learning_rate=0.05,
                     n_estimators=600,
                     subsample=0.8,
                     colsample_bytree=0.8,
                     random_state=42,
                     n_jobs=1,            # avoid nested parallelism with GridSearchCV
                     verbosity=0,
                 )),
])

param_grid = {
    "model__reg_lambda":       [0, 1, 5],
    "model__max_depth":        [4, 6],
    "model__min_child_weight": [1, 5],
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipe, param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)
search.fit(X, y)

print(f"Best CV RMSE (log-price): {-search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

results = pd.DataFrame(search.cv_results_)
top = (
    results[[
        "param_model__reg_lambda",
        "param_model__max_depth",
        "param_model__min_child_weight",
        "mean_test_score", "std_test_score",
    ]]
    .assign(mean_rmse=lambda d: -d["mean_test_score"])
    .sort_values("mean_rmse")
    .drop(columns=["mean_test_score"])
)
print("\nAll configs:")
print(top.to_string(index=False))

pipe = search.best_estimator_

In [ ]:
test_pred = np.expm1(pipe.predict(X_test))

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = sample.copy()
submission["SalePrice"] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)